In [ ]:
# Colab: Runtime → Change runtime type → GPU (T4). CUDA thường là 12.x; nếu lỗi wheel, đổi cu124 → cu121.
# Kiểm tra sau khi chạy: import torch; print(torch.cuda.get_device_name(0))

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

!pip install -q mamba-ssm
!pip install -q numpy scikit-learn

!pip install -q tensorboard einops ninja


## Nhiệm vụ notebook này

Luồng **end-to-end**: nhận data Stage A → train từng backbone thời gian → **so sánh hai nhóm metric**:

| Nhóm | Mục tiêu | Metric validation (in sau khi train) |
|------|-----------|--------------------------------------|
| **Risk** | Xác suất va chạm 0.5s / 1s / 2s | AP@0.5s, AP@1s, AP@2s, AUC@1s |
| **Quỹ đạo** | Dự đoán **H bước** tương lai `(x, y, yaw)` trong world (từ cùng `h_T`) | RMSE toàn cục, **ADE_xy / FDE_xy** (m), RMSE yaw (rad) |

1. **Data:** `data/stage_a_experiment/` (`index.jsonl` + `*.npz`; `ego_state` + nhãn `risk_*`).
2. **Train:** Mamba / GRU / LSTM / Transformer — PointPillars **frozen**; train reducer + temporal + **RiskHead + TrajectoryHead**; loss = focal BCE (risk) + trọng số × SmoothL1 (quỹ đạo).
3. **Output:** hai bảng in ra console + `summary.json` + TensorBoard + **file trọng số `.pt` theo từng backbone** (mặc định trong `runs/stage_a_compare/weights/`): `{tên}_stage_a_compare_<timestamp>.pt` và `{tên}_stage_a_compare.pt` (bản *latest* ghi đè mỗi lần train lại backbone đó).

**Chuẩn bị:** `python run_datagen_preset.py experiment`; checkpoint `PointPillars_module/pretrained/epoch_160.pt` hoặc `.pth`.


In [ ]:
# --- Bước 1–2: đường dẫn + kiểm tra data & checkpoint (chưa train) ---
import os
import sys

REPO_ROOT = "/content/Pipeline"  # đổi nếu clone Drive: /content/drive/MyDrive/.../Pipeline
DATA_ROOT = os.path.join(REPO_ROOT, "data", "stage_a_experiment")
PRETRAINED_DIR = os.path.join(REPO_ROOT, "PointPillars_module", "pretrained")
CKPT = None
for _name in ("epoch_160.pt", "epoch_160.pth"):
    _p = os.path.join(PRETRAINED_DIR, _name)
    if os.path.isfile(_p):
        CKPT = _p
        break
LOG_ROOT = os.path.join(REPO_ROOT, "runs", "stage_a_compare")

# Huấn luyện — chỉnh tại đây (áp dụng cho mọi backbone)
EPOCHS = 3
BATCH_SIZE = 4
LR = 3e-4
MODELS = ("mamba", "gru", "lstm", "transformer")
SEED = 0
# H tương lai (20 Hz): 10 frame = 0.5 s — khớp mặc định RiskDataset / TrajectoryHead
TRAJ_HORIZON = 10
# Trọng số loss quỹ đạo so với risk (focal BCE)
TRAJ_LOSS_WEIGHT = 0.5
# None → lưu dưới LOG_ROOT/weights ; hoặc set chuỗi path tùy ý
WEIGHTS_DIR = None

for path in (REPO_ROOT, os.path.join(REPO_ROOT, "PointPillars_module"), os.path.join(REPO_ROOT, "create_dataset_module")):
    if path not in sys.path:
        sys.path.insert(0, path)
os.chdir(REPO_ROOT)

assert os.path.isdir(DATA_ROOT), f"Thiếu data: {DATA_ROOT} — chạy `python run_datagen_preset.py experiment` rồi upload."
assert os.path.isfile(os.path.join(DATA_ROOT, "index.jsonl")), f"Thiếu index.jsonl trong {DATA_ROOT}"
assert CKPT is not None, (
    f"Thiếu checkpoint trong {PRETRAINED_DIR}. "
    f"Cần epoch_160.pt hoặc epoch_160.pth."
)

print("OK — data:", DATA_ROOT)
print("OK — ckpt:", CKPT)
print("OK — sẽ train + so sánh:", MODELS)


In [ ]:
# --- Bước 3: train từng mô hình + in bảng so sánh (cùng một lần gọi) ---
from train_stage_a_compare import run_experiment

results = run_experiment(
    data_root=DATA_ROOT,
    ckpt_path=CKPT,
    models=MODELS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    log_root=LOG_ROOT,
    device=None,
    seed=SEED,
    traj_horizon=TRAJ_HORIZON,
    traj_loss_weight=TRAJ_LOSS_WEIGHT,
    weights_dir=WEIGHTS_DIR,
    save_weights=True,
    mamba_backend="auto",
)

**Sau train:** TensorBoard `%tensorboard --logdir runs/stage_a_compare`. **`summary.json`**: metrics + `checkpoint_pt` / `checkpoint_pt_latest`. Thư mục **`weights/`**: file `.pt` đầy đủ `model_state_dict` (CPU) + metadata (`backbone`, `pp_ckpt_path`, `traj_horizon`, …).

**Graph:** `pts_seq` → PointPillars (frozen) → reducer → temporal → **`h_T`** → **RiskHead** (3 logits) + **TrajectoryHead** (`H×3`).

Cell dưới in lại `summary.json` (risk + quỹ đạo + đường dẫn `.pt`).


In [ ]:
# --- Bước 4 (tuỳ chọn): đọc lại summary.json — hai bảng ---
import json
from pathlib import Path

summary_path = Path(LOG_ROOT) / "summary.json"
if not summary_path.is_file():
    print("Chưa có summary.json — chạy cell train trước.")
else:
    with summary_path.open(encoding="utf-8") as f:
        summary = json.load(f)
    print("File:", summary_path)
    print("\n[Risk]")
    print(f"{'model':<12} {'AP@0.5s':>8} {'AP@1s':>8} {'AP@2s':>8} {'AUC@1s':>8}")
    for name, m in summary.items():
        print(
            f"{name:<12} {m.get('ap_risk_05s', 'nan'):>8} {m.get('ap_risk_1s', 'nan'):>8} "
            f"{m.get('ap_risk_2s', 'nan'):>8} {m.get('auc_risk_1s', 'nan'):>8}"
        )
    print("\n[Trajectory H={} steps]".format(TRAJ_HORIZON))
    print(f"{'model':<12} {'RMSE_all':>10} {'ADE_xy':>10} {'FDE_xy':>10} {'RMSE_yaw':>10}")
    for name, m in summary.items():
        print(
            f"{name:<12} {m.get('traj_rmse_all', 'nan'):>10} {m.get('traj_ade_xy_m', 'nan'):>10} "
            f"{m.get('traj_fde_xy_m', 'nan'):>10} {m.get('traj_rmse_yaw_rad', 'nan'):>10}"
        )
    print("\n[Checkpoints .pt]")
    for name, m in summary.items():
        print(f"  {name}: {m.get('checkpoint_pt', '—')}")
        print(f"         (latest) {m.get('checkpoint_pt_latest', '—')}")